# Grader Service – Demo Notebook

Sends requests to the running service at `http://localhost:8000` and displays the results.  
Make sure `docker compose up --build` has finished before running the cells.

In [16]:
import json
import requests
from IPython.display import display
import pandas as pd

BASE_URL = "http://localhost:8000"

## 1. Health check

In [17]:
response = requests.get(f"{BASE_URL}/health", timeout=5)
print(f"Status: {response.status_code}")
print(response.json())

Status: 200
{'status': 'ok'}


## 2. Build a grading request

Edit the question, solution, rubrics, and student answers here.

In [18]:
payload = {
    "question_id": "q42",
    "question": "What causes seasons on Earth?",
    "solution": "Seasons are caused by the tilt of Earth's axis, not its distance from the Sun.",
    "rubrics": [
        {"id": "r1", "text": "The answer mentions the tilt of Earth's axis."},
        {"id": "r2", "text": "The answer rejects distance from the Sun as the primary cause."},
        {"id": "r3", "text": "The answer is scientifically accurate overall.", "is_ternary": True},
    ],
    "student_answers": [
        {"id": "a1", "text": "Because the Earth is tilted on its axis."},
        {"id": "a2", "text": "The Earth gets closer to the Sun in summer."},
        {"id": "a3", "text": "The tilt of the Earth causes different parts to receive more sunlight."},
    ],
}

## 3. Send the request and display raw response

In [21]:
response = requests.post(f"{BASE_URL}/grade", json=payload, timeout=30)
response.raise_for_status()
result = response.json()
print(json.dumps(result, indent=2))

{
  "question_id": "q42",
  "results": [
    {
      "answer_id": "a1",
      "rubric_decisions": [
        {
          "rubric_id": "r1",
          "decision": "no"
        },
        {
          "rubric_id": "r2",
          "decision": "yes"
        },
        {
          "rubric_id": "r3",
          "decision": "half"
        }
      ]
    },
    {
      "answer_id": "a2",
      "rubric_decisions": [
        {
          "rubric_id": "r1",
          "decision": "yes"
        },
        {
          "rubric_id": "r2",
          "decision": "yes"
        },
        {
          "rubric_id": "r3",
          "decision": "yes"
        }
      ]
    },
    {
      "answer_id": "a3",
      "rubric_decisions": [
        {
          "rubric_id": "r1",
          "decision": "yes"
        },
        {
          "rubric_id": "r2",
          "decision": "yes"
        },
        {
          "rubric_id": "r3",
          "decision": "half"
        }
      ]
    }
  ]
}


## 4. Results as a table

In [22]:
# Build lookup maps for readable labels
answer_text = {a["id"]: a["text"] for a in payload["student_answers"]}
rubric_text  = {r["id"]: r["text"] for r in payload["rubrics"]}

rows = []
for answer_result in result["results"]:
    aid = answer_result["answer_id"]
    for rd in answer_result["rubric_decisions"]:
        rows.append({
            "answer_id":   aid,
            "answer_text": answer_text[aid],
            "rubric_id":   rd["rubric_id"],
            "rubric_text": rubric_text[rd["rubric_id"]],
            "decision":    rd["decision"],
        })

df = pd.DataFrame(rows)

# Colour-code decisions for readability
COLOURS = {"yes": "background-color: #c6efce", "no": "background-color: #ffc7ce", "half": "background-color: #ffeb9c"}

display(
    df.style
      .map(lambda v: COLOURS.get(v, ""), subset=["decision"])
      .set_properties(**{"text-align": "left"})
      .hide(axis="index")
)

answer_id,answer_text,rubric_id,rubric_text,decision
a1,Because the Earth is tilted on its axis.,r1,The answer mentions the tilt of Earth's axis.,no
a1,Because the Earth is tilted on its axis.,r2,The answer rejects distance from the Sun as the primary cause.,yes
a1,Because the Earth is tilted on its axis.,r3,The answer is scientifically accurate overall.,half
a2,The Earth gets closer to the Sun in summer.,r1,The answer mentions the tilt of Earth's axis.,yes
a2,The Earth gets closer to the Sun in summer.,r2,The answer rejects distance from the Sun as the primary cause.,yes
a2,The Earth gets closer to the Sun in summer.,r3,The answer is scientifically accurate overall.,yes
a3,The tilt of the Earth causes different parts to receive more sunlight.,r1,The answer mentions the tilt of Earth's axis.,yes
a3,The tilt of the Earth causes different parts to receive more sunlight.,r2,The answer rejects distance from the Sun as the primary cause.,yes
a3,The tilt of the Earth causes different parts to receive more sunlight.,r3,The answer is scientifically accurate overall.,half
